# GoEmotions - EDA and data preparation

First look at the data with basic tools only (pandas, matplotlib, counting), then the data preparation used by every model.
Dataset: GoEmotions, *simplified* version (Reddit comments, 27 emotions + neutral, several labels possible per comment).

Run from `notebooks/`. Figures are saved to `results/eda_simple/`, the prepared splits to `data/processed/` (not committed: run this notebook once before the modelling notebooks).

In [ ]:
%matplotlib inline
import os
import re
import emoji
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

FIG_DIR = os.path.join("..", "results", "eda_simple")
os.makedirs(FIG_DIR, exist_ok = True)

## 1. Load the data

In [ ]:
ds = load_dataset("google-research-datasets/go_emotions", "simplified")
label_names = ds["train"].features["labels"].feature.names

df_train = ds["train"].to_pandas()
df_val = ds["validation"].to_pandas()
df_test = ds["test"].to_pandas()

print("Number of labels:", len(label_names))
print("Train:", df_train.shape)
print("Validation:", df_val.shape)
print("Test:", df_test.shape)
df_train.head()

In [ ]:
print(label_names)

## 2. Basic checks

In [ ]:
for name, df in [("Train", df_train), ("Validation", df_val), ("Test", df_test)]:
    print(name)
    print("  Missing values:", df.isna().sum().sum())
    print("  Empty texts:", (df["text"].str.strip() == "").sum())
    print("  Comments with no label:", (df["labels"].apply(len) == 0).sum())

In [ ]:
# replace the label ids by their names to make the data readable
df_train["label_names"] = df_train["labels"].apply(lambda l: [label_names[i] for i in l])
df_train[["text", "label_names"]].head(10)

## 3. How often does each emotion appear?

In [ ]:
all_labels_train = [lab for l in df_train["label_names"] for lab in l]
label_counts = pd.Series(Counter(all_labels_train)).sort_values(ascending = False)

label_table = pd.DataFrame({"count": label_counts,
                            "percent_of_comments": (100 * label_counts / len(df_train)).round(2)})
label_table

In [ ]:
fig, ax = plt.subplots(figsize = (10, 4))
ax.bar(label_counts.index, label_counts.values, color = "#2a78d6")
ax.set_title("Number of train comments per label")
ax.set_ylabel("Comments")
plt.xticks(rotation = 90)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "label_counts.png"))
plt.show()

In [ ]:
print("Most frequent label:", label_counts.index[0], label_counts.iloc[0])
print("Least frequent label:", label_counts.index[-1], label_counts.iloc[-1])
print("Ratio most / least frequent:", round(label_counts.iloc[0] / label_counts.iloc[-1], 1))
label_counts_no_neutral = label_counts.drop("neutral")
print("Ratio most / least frequent (without neutral):", round(label_counts_no_neutral.iloc[0] / label_counts_no_neutral.iloc[-1], 1))

## 4. How many labels per comment?

In [ ]:
df_train["nb_labels"] = df_train["labels"].apply(len)
nb_labels_counts = df_train["nb_labels"].value_counts().sort_index()
print(nb_labels_counts)
print("Percent of comments with one label:", round(100 * (df_train["nb_labels"] == 1).mean(), 1))

In [ ]:
fig, ax = plt.subplots(figsize = (5, 3.5))
ax.bar(nb_labels_counts.index.astype(str), nb_labels_counts.values, color = "#2a78d6")
ax.set_title("Labels per comment (train)")
ax.set_xlabel("Number of labels")
ax.set_ylabel("Comments")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "labels_per_comment.png"))
plt.show()

## 5. Which labels appear together?

In [ ]:
pairs = Counter()
for l in df_train["label_names"]:
    if len(l) >= 2:
        for i in range(len(l)):
            for j in range(i + 1, len(l)):
                pairs[tuple(sorted([l[i], l[j]]))] += 1

top_pairs = pd.DataFrame(pairs.most_common(10), columns = ["pair", "count"])
top_pairs

## 6. Length of the comments

Words are counted by splitting on spaces (`str.split()`), not with an NLP tokenizer, so these are approximate word counts, not model tokens.

In [ ]:
df_train["nb_words"] = df_train["text"].str.split().apply(len)
df_train["nb_words"].describe()

In [ ]:
fig, ax = plt.subplots(figsize = (6, 3.5))
ax.hist(df_train["nb_words"], bins = 30, color = "#2a78d6", edgecolor = "white")
ax.set_title("Words per comment (train)")
ax.set_xlabel("Number of words")
ax.set_ylabel("Comments")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "comment_length.png"))
plt.show()

In [ ]:
# average length for each label
df_exploded = df_train.explode("label_names")
length_by_label = df_exploded.groupby("label_names")["nb_words"].mean().round(1).sort_values()
print("Shortest on average:")
print(length_by_label.head(3))
print("Longest on average:")
print(length_by_label.tail(3))

## 7. Some examples

In [ ]:
for emotion in ["gratitude", "anger", "sadness", "curiosity", "neutral"]:
    examples = df_exploded[df_exploded["label_names"] == emotion]["text"].sample(3, random_state = 0)
    print("---", emotion)
    for t in examples:
        print("  ", t)

## 8. Most frequent words

In [ ]:
def tokenize(text):
    words = re.findall(r"[a-z']+", text.lower())
    return [w for w in words if w not in ENGLISH_STOP_WORDS and len(w) > 2]

all_words = Counter()
for t in df_train["text"]:
    all_words.update(tokenize(t))
print("Most frequent words overall:")
print(all_words.most_common(15))

In [ ]:
for emotion in ["gratitude", "anger", "sadness", "amusement", "fear"]:
    words_emotion = Counter()
    for t in df_exploded[df_exploded["label_names"] == emotion]["text"]:
        words_emotion.update(tokenize(t))
    print(emotion, ":", [w for w, n in words_emotion.most_common(10)])

In [ ]:
# punctuation: share of comments with "?" or "!" for a few labels
df_exploded["has_question"] = df_exploded["text"].str.contains("\\?")
df_exploded["has_exclamation"] = df_exploded["text"].str.contains("!")
punct = df_exploded.groupby("label_names")[["has_question", "has_exclamation"]].mean().round(2)
punct.loc[["curiosity", "confusion", "excitement", "gratitude", "neutral", "sadness"]]

## 9. Duplicates between splits

Texts are compared **after lowercasing**, so "Thank you." and "thank you." count as the same text.

In [ ]:
train_texts = set(df_train["text"].str.lower())
val_texts = set(df_val["text"].str.lower())

print("Duplicate texts inside train:", df_train["text"].str.lower().duplicated().sum())
print("Validation texts also in train:", df_val["text"].str.lower().isin(train_texts).sum())
print("Test texts also in train:", df_test["text"].str.lower().isin(train_texts).sum())
print("Test texts also in validation:", df_test["text"].str.lower().isin(val_texts).sum())

In [ ]:
df_test[df_test["text"].str.lower().isin(train_texts)]["text"].head(10)

## 10. Do the three splits have the same label distribution?

In [ ]:
def label_percent(df):
    names = [label_names[i] for l in df["labels"] for i in l]
    return 100 * pd.Series(Counter(names)) / len(df)

split_compare = pd.DataFrame({"train": label_percent(df_train),
                              "validation": label_percent(df_val),
                              "test": label_percent(df_test)}).round(2)
split_compare = split_compare.sort_values("train", ascending = False)
split_compare.head(10)

In [ ]:
print("Largest difference between splits (percentage points):", round((split_compare.max(axis = 1) - split_compare.min(axis = 1)).max(), 2))

# Data preparation

The official splits are kept, with these cleaning choices, agreed by the team:

1. **Emojis converted to text** with the `emoji` library (😂 becomes `:face_with_tears_of_joy:`). About 2% of comments contain one, often very emotional, and both models would otherwise lose them: the transformer sees an unknown token `[UNK]`, and the default TF-IDF drops them.
2. **Letters repeated more than 3 times reduced to 3** ("fuuuuuuuuck" becomes "fuuuck", "OHHHHHHHHH" becomes "OHHH"). The intensity is kept and the word stays readable, instead of being cut into dozens of meaningless tokens.
3. **Duplicates inside train:** comments with the same text (after lowercasing) are merged into one row, whose labels are the union of the labels of the copies.
4. **Validation and test texts already seen in train** are removed, so a model is never scored on a text it was trained on.
5. **Test texts also in validation** are removed from test, because validation is used to tune the thresholds.

Steps 1 and 2 run **before** the duplicate removal, because cleaning can create new duplicates. Both are applied the same way to every split, so the baseline and the transformer read exactly the same texts.

Nothing else is changed: case, punctuation and the `[NAME]` placeholders are kept (the transformer notebook adds `[NAME]` as a special token on its side).

## 11. Text cleaning: emojis and repeated letters

In [ ]:
def clean_text(text):
    # 1. emojis -> text, with spaces around so they stay separate words
    if emoji.emoji_count(text) > 0:
        text = emoji.demojize(text, delimiters = (" :", ": "))
        # remove the double spaces this creates
        text = re.sub(r" {2,}", " ", text).strip()
    # 2. a letter repeated more than 3 times -> 3 times
    text = re.sub(r"([A-Za-z])\1{3,}", r"\1\1\1", text)
    return text

for text in ["lol😂😂 I love it 😍", "fuuuuuuuuck OHHHHHHHHH", "WHY THE FUCK IS BAYLESS ISOING", "1000000 dollars"]:
    print(text, " -> ", clean_text(text))

In [ ]:
df_train_cleantext = df_train[["id", "text", "labels"]].copy()
df_val_cleantext = df_val[["id", "text", "labels"]].copy()
df_test_cleantext = df_test[["id", "text", "labels"]].copy()

for name, df in [("Train", df_train_cleantext), ("Validation", df_val_cleantext), ("Test", df_test_cleantext)]:
    df["text_raw"] = df["text"]
    df["text"] = df["text_raw"].apply(clean_text)
    has_emoji = df["text_raw"].apply(lambda t: emoji.emoji_count(t) > 0)
    # re.search, not str.contains: the Arrow string engine of pandas 3 does not support \1
    has_repeat = df["text_raw"].apply(lambda t: re.search(r"([A-Za-z])\1{3,}", t) is not None)
    print(name, "- with emojis:", round(100 * has_emoji.mean(), 2), "%",
          "| with repeated letters:", round(100 * has_repeat.mean(), 2), "%",
          "| texts changed:", (df["text"] != df["text_raw"]).sum())

## 12. Remove duplicates

In [ ]:
def normalize(text):
    return text.lower()

df_train_clean = df_train_cleantext.copy()
df_train_clean["text_norm"] = df_train_clean["text"].apply(normalize)

# merge duplicate texts in train (after cleaning, so new duplicates are caught too): keep the first id and text, take the union of the labels
df_train_clean = (df_train_clean
                  .groupby("text_norm", sort = False)
                  .agg(id = ("id", "first"), text = ("text", "first"), text_raw = ("text_raw", "first"),
                       labels = ("labels", lambda ls: sorted(set(i for l in ls for i in l))))
                  .reset_index())

print("Train rows before:", len(df_train), "after:", len(df_train_clean), "removed:", len(df_train) - len(df_train_clean))

In [ ]:
train_norm = set(df_train_clean["text_norm"])

df_val_clean = df_val_cleantext.copy()
df_val_clean["text_norm"] = df_val_clean["text"].apply(normalize)
df_val_clean = df_val_clean[~df_val_clean["text_norm"].isin(train_norm)].reset_index(drop = True)

val_norm = set(df_val_clean["text_norm"])

df_test_clean = df_test_cleantext.copy()
df_test_clean["text_norm"] = df_test_clean["text"].apply(normalize)
df_test_clean = df_test_clean[~df_test_clean["text_norm"].isin(train_norm)]
df_test_clean = df_test_clean[~df_test_clean["text_norm"].isin(val_norm)].reset_index(drop = True)

print("Validation rows before:", len(df_val), "after:", len(df_val_clean), "removed:", len(df_val) - len(df_val_clean))
print("Test rows before:", len(df_test), "after:", len(df_test_clean), "removed:", len(df_test) - len(df_test_clean))

In [ ]:
# checks: no text shared between splits anymore
print("Validation texts in train:", df_val_clean["text_norm"].isin(train_norm).sum())
print("Test texts in train:", df_test_clean["text_norm"].isin(train_norm).sum())
print("Test texts in validation:", df_test_clean["text_norm"].isin(val_norm).sum())
print("Duplicate texts inside train:", df_train_clean["text_norm"].duplicated().sum())

### 12.1. Automated data leakage checks

We use assertions to ensure that no normalized text
is shared between the training, validation and test sets.
These checks prevent the pipeline from continuing if
data leakage is detected.

In [ ]:
assert not df_val_clean["text_norm"].isin(train_norm).any()
assert not df_test_clean["text_norm"].isin(train_norm).any()
assert not df_test_clean["text_norm"].isin(val_norm).any()

assert not df_train_clean["text_norm"].duplicated().any()

print("All leakage checks passed.")

In [ ]:
for name, df in [
    ("Validation", df_val_clean),
    ("Test", df_test_clean)
]:
    print(name, "- Internal duplicates:",
          df["text_norm"].duplicated().sum())

Internal duplicates in validation and test are reported above. They do not create overlap between splits but may affect evaluation weights. We keep the existing split policy to maintain consistency with baseline experiments.

### 12.2. Duplicate label consistency

Identical comments may have different emotion annotations.
We analyze these disagreements to assess the impact
of merging their labels.

In [ ]:
# Compare the original and current normalization policies separately.
for name, source in [("Raw", df_train), ("After text cleaning", df_train_cleantext)]:
    duplicate_analysis = source[["text", "labels"]].copy()
    duplicate_analysis["text_norm"] = duplicate_analysis["text"].apply(normalize)
    label_variations = duplicate_analysis.groupby("text_norm")["labels"].apply(
        lambda labels: len(set(tuple(sorted(l)) for l in labels))
    )
    print(name, "- Duplicate groups with different labels:", (label_variations > 1).sum())

In [ ]:
neutral_id = label_names.index("neutral")

def has_neutral_conflict(labels):
    return neutral_id in labels and len(labels) > 1

before = df_train["labels"].apply(has_neutral_conflict).sum()

after = df_train_clean["labels"].apply(
    has_neutral_conflict
).sum()

print("Neutral + other emotions before cleaning:", before)
print("Neutral + other emotions after cleaning:", after)

**Interpretation**

The previous lowercase-only pipeline found 61 groups with differing annotations and 1,396 / 1,412 Neutral co-occurrences before / after merging. The current pipeline additionally normalizes emojis and repeated letters; the checks above must be rerun to establish its figures.

Neutral co-occurrences already exist in the original annotations. We retain label unions rather than removing Neutral arbitrarily, while acknowledging that merging may introduce ambiguity.

## 13. Labels at three levels

The 28 labels are mapped to the 6 Ekman emotions and to 3 sentiment groups with the official mappings from the GoEmotions documentation (`ekman_mapping.json`, `sentiment_mapping.json` in google-research/goemotions). Neutral stays its own class. A comment gets a group when at least one of its emotions belongs to it, so every level stays multi-label.

In [ ]:
ekman_mapping = {
    "anger": ["anger", "annoyance", "disapproval"],
    "disgust": ["disgust"],
    "fear": ["fear", "nervousness"],
    "joy": ["joy", "amusement", "approval", "excitement", "gratitude", "love", "optimism", "relief", "pride", "admiration", "desire", "caring"],
    "sadness": ["sadness", "disappointment", "embarrassment", "grief", "remorse"],
    "surprise": ["surprise", "realization", "confusion", "curiosity"],
}
sentiment_mapping = {
    "positive": ["amusement", "excitement", "joy", "love", "desire", "optimism", "caring", "pride", "admiration", "gratitude", "relief", "approval"],
    "negative": ["fear", "nervousness", "remorse", "embarrassment", "disappointment", "sadness", "grief", "disgust", "anger", "annoyance", "disapproval"],
    "ambiguous": ["realization", "surprise", "curiosity", "confusion"],
}

# every emotion must appear in exactly one group of each mapping
emotions = [l for l in label_names if l != "neutral"]
for name, mapping in [("Ekman", ekman_mapping), ("Sentiment", sentiment_mapping)]:
    mapped = [e for group in mapping.values() for e in group]
    print(name, "- all emotions mapped once:", sorted(mapped) == sorted(emotions))

In [ ]:
def to_group(names, mapping):
    groups = set()
    for n in names:
        if n == "neutral":
            groups.add("neutral")
        for g, members in mapping.items():
            if n in members:
                groups.add(g)
    return sorted(groups)

for df in [df_train_clean, df_val_clean, df_test_clean]:
    df["labels_28"] = df["labels"].apply(lambda l: [label_names[i] for i in l])
    df["labels_ekman"] = df["labels_28"].apply(lambda l: to_group(l, ekman_mapping))
    df["labels_sentiment"] = df["labels_28"].apply(lambda l: to_group(l, sentiment_mapping))

df_train_clean[["text", "labels_28", "labels_ekman", "labels_sentiment"]].head(8)

In [ ]:
def group_counts(df, column):
    return pd.Series(Counter(g for l in df[column] for g in l)).sort_values(ascending = False)

print("Ekman (train comments per group):")
print(group_counts(df_train_clean, "labels_ekman"))
print("Sentiment (train comments per group):")
print(group_counts(df_train_clean, "labels_sentiment"))

## 14. Save the prepared splits

In [ ]:
OUT_DIR = os.path.join("..", "data", "processed")
os.makedirs(OUT_DIR, exist_ok = True)

# text = cleaned text (use this one in every model), text_raw = original text, kept for error analysis
columns = ["id", "text", "text_raw", "labels_28", "labels_ekman", "labels_sentiment"]
df_train_clean[columns].to_parquet(os.path.join(OUT_DIR, "train.parquet"), index = False)
df_val_clean[columns].to_parquet(os.path.join(OUT_DIR, "validation.parquet"), index = False)
df_test_clean[columns].to_parquet(os.path.join(OUT_DIR, "test.parquet"), index = False)

print("Saved to", OUT_DIR)
print("Train:", len(df_train_clean), "Validation:", len(df_val_clean), "Test:", len(df_test_clean))

In [ ]:
for split in ["train", "validation", "test"]:
    path = os.path.join(OUT_DIR, f"{split}.parquet")
    df = pd.read_parquet(path)
    assert list(df.columns) == columns
    assert not df.isna().any().any()
    for col in ["labels_28", "labels_ekman", "labels_sentiment"]:
        assert df[col].apply(len).gt(0).all()
    print(f"{split}: {df.shape} - OK")

To load them in another notebook:

```python
df_train = pd.read_parquet(os.path.join("..", "data", "processed", "train.parquet"))
```

## 15. Summary

**EDA**
- **Size:** 43,410 train / 5,426 validation / 5,427 test comments, 28 labels (27 emotions + neutral). No missing values, no empty texts and no comment without a label, in all three splits.
- **Strong imbalance:** `neutral` is on 32.8% of train comments. The most frequent emotion (admiration, 4,130) appears 54 times more often than the rarest (grief, 77). Pride, relief, nervousness and grief each have fewer than 200 train examples.
- **Mostly single-label:** 83.6% of comments have one label, 15.1% have two. The labels that go together are close emotions: anger + annoyance, confusion + curiosity, disappointment + sadness, admiration + gratitude.
- **Short texts:** 12 words on median, 33 at most (counted by splitting on spaces, not tokenizer tokens). Length barely changes between emotions, so it is not a useful feature on its own.
- **Words and punctuation carry a lot of signal:** "thanks" for gratitude, "lol/haha" for amusement, "scared/afraid" for fear; 81% of curiosity comments contain a `?` and 42% of excitement comments a `!`. Punctuation is preserved during data preparation; its use by TF-IDF depends on the vectorizer configuration.
- **Labels are noisy:** some examples look wrong or ambiguous, so part of the model errors will be annotation errors.
- **Duplicates (compared after lowercasing):** 220 repeated texts inside train, 49 validation and 42 test texts also in train, and 14 test texts also in validation. They are short generic comments such as "Thank you.".
- **Splits are consistent:** label shares differ by less than 1 percentage point between train, validation and test.

**Data preparation**
- Emojis converted to text (about 2% of comments) and letters repeated more than 3 times reduced to 3 (about 1%), on all splits, before removing duplicates: 1,212 train, 153 validation and 158 test texts changed. The original text is kept in `text_raw`.
- Train duplicates merged with the union of their labels: 43,410 → 43,188 rows (the cleaning created 2 new duplicates).
- Validation and test texts already seen in train removed, and test texts seen in validation removed: validation 5,426 → 5,377, test 5,427 → 5,379. No text is shared between splits anymore.
- Labels built at three levels (28 labels, Ekman 6 + neutral, sentiment 3 + neutral) with the official GoEmotions mappings, saved to `data/processed/`.

- Automated assertions, duplicate annotation checks and Parquet reload checks are included; rerun them on the current preprocessing pipeline to verify the final results.

**Consequence for modelling:** due to the strong class imbalance, macro-F1 will be our primary metric, complemented by micro-F1 and per-class metrics. Accuracy is insufficient as a primary metric because it hides poor performance on the rare classes. Class weighting and per-class decision thresholds will be investigated during model development.